In [7]:
import os
import pandas as pd

# Define the directory containing the CSV files
directory = '/mnt/scratch_lustre/barthelx/Masrur/Projects/Data_imputation/Continuous_Data'
output_directory = '/mnt/scratch_lustre/barthelx/Masrur/Projects/Data_imputation/Summery_results2'

# Create the output directory if it does not exist
os.makedirs(output_directory, exist_ok=True)

# Function to extract study site from filename
def extract_study_site(filename):
    parts = filename.split('_')
    study_site = parts[1]  # Assuming the study site is the second word in the filename
    return study_site

# Function to rename columns by extracting the variable name
def rename_columns(df):
    renamed_columns = {}
    study_site = None
    for col in df.columns:
        if col != 'datetime':
            parts = col.split('_')
            if len(parts) > 1:
                var_name = parts[0]
                renamed_columns[col] = var_name
                if not study_site:
                    study_site = parts[1]  # Assuming all columns have the same study site
    df.rename(columns=renamed_columns, inplace=True)
    return study_site

# Function to calculate missing values and percentages
def calculate_missing_values(df):
    missing_values = df.isnull().sum()
    total_values = len(df) * len(df.columns)
    missing_percent = (missing_values / total_values) * 100
    return missing_values, missing_percent

# Initialize dataframes to store missing value reports
site_missing_values_df = pd.DataFrame()
hourly_missing_values_df = pd.DataFrame()
yearly_missing_values_df = pd.DataFrame()

# Traverse the directory and process each CSV file
for filename in os.listdir(directory):
    if filename.endswith('.csv'):
        filepath = os.path.join(directory, filename)

        # Load the CSV file into a DataFrame
        df = pd.read_csv(filepath)

        # Convert datetime column to datetime type and sort by datetime
        df['datetime'] = pd.to_datetime(df['datetime'])
        df.sort_values(by='datetime', inplace=True)

        # Extract study site from filename
        study_site = extract_study_site(filename)

        # Rename columns
        rename_columns(df)

        # Set the datetime column as the index
        df.set_index('datetime', inplace=True)

        # Calculate missing values for each study site
        missing_values, missing_percent = calculate_missing_values(df)
        site_missing_values_df = site_missing_values_df.append({
            'StudySite': study_site,
            'TotalValues': len(df) * len(df.columns),
            **missing_percent.to_dict()
        }, ignore_index=True)

        # Calculate missing values per hour for each study site
        hourly_groups = df.groupby(df.index.hour)
        for hour, group in hourly_groups:
            missing_values, missing_percent = calculate_missing_values(group)
            hourly_missing_values_df = hourly_missing_values_df.append({
                'StudySite': study_site,
                'Hour': hour,
                'TotalValues': len(group) * len(group.columns),
                **missing_percent.to_dict()
            }, ignore_index=True)

        # Calculate missing values per year for each study site
        yearly_groups = df.groupby(df.index.year)
        for year, group in yearly_groups:
            missing_values, missing_percent = calculate_missing_values(group)
            yearly_missing_values_df = yearly_missing_values_df.append({
                'StudySite': study_site,
                'Year': year,
                'TotalValues': len(group) * len(group.columns),
                **missing_percent.to_dict()
            }, ignore_index=True)

# Save the missing value reports to CSV files
site_missing_values_df.to_csv(os.path.join(output_directory, 'Site_Missing_Values_Report.csv'), index=False)
hourly_missing_values_df.to_csv(os.path.join(output_directory, 'Hourly_Missing_Values_Report.csv'), index=False)
yearly_missing_values_df.to_csv(os.path.join(output_directory, 'Yearly_Missing_Values_Report.csv'), index=False)

print("Missing value reports generated and saved.")

Missing value reports generated and saved.


In [4]:
import os
import pandas as pd

# Define the directory containing the CSV files
directory = '/home/ahmedmas/Projects/Data_imputation/Processed_Data'
output_directory = '/home/ahmedmas/Projects/Data_imputation/Summery_results_processed2'

# Create the output directory if it does not exist
os.makedirs(output_directory, exist_ok=True)

# Function to extract study site from filename
def extract_study_site(filename):
    parts = filename.split('_')
    study_site = parts[1]  # Assuming the study site is the second word in the filename
    return study_site

# Function to rename columns by extracting the variable name
def rename_columns(df):
    renamed_columns = {}
    study_site = None
    for col in df.columns:
        if col != 'datetime':
            parts = col.split('_')
            if len(parts) > 1:
                var_name = parts[0]
                renamed_columns[col] = var_name
                if not study_site:
                    study_site = parts[1]  # Assuming all columns have the same study site
    df.rename(columns=renamed_columns, inplace=True)
    return study_site

# Function to calculate missing values and percentages
def calculate_missing_values(df):
    missing_values = df.isnull().sum()
    total_values = len(df)
    missing_percent = (missing_values / total_values) * 100
    return missing_values, missing_percent, total_values

# Initialize dataframes to store missing value reports
site_missing_values_df = pd.DataFrame()
hourly_missing_values_df = pd.DataFrame()
yearly_missing_values_df = pd.DataFrame()

# Traverse the directory and process each CSV file
for filename in os.listdir(directory):
    if filename.endswith('.csv'):
        filepath = os.path.join(directory, filename)

        # Load the CSV file into a DataFrame
        df = pd.read_csv(filepath)

        # Convert datetime column to datetime type and sort by datetime
        df['datetime'] = pd.to_datetime(df['datetime'])
        df.sort_values(by='datetime', inplace=True)

        # Extract study site from filename
        study_site = extract_study_site(filename)

        # Rename columns
        rename_columns(df)

        # Set the datetime column as the index
        df.set_index('datetime', inplace=True)

        # Calculate missing values for each study site
        missing_values, missing_percent, total_values = calculate_missing_values(df)
        site_missing_values_df = site_missing_values_df.append({
            'StudySite': study_site,
            'TotalValues': total_values,
            **{col: missing_percent[col] for col in df.columns}
        }, ignore_index=True)

        # Calculate missing values per hour for each study site
        hourly_groups = df.groupby(df.index.hour)
        for hour, group in hourly_groups:
            missing_values, missing_percent, total_values = calculate_missing_values(group)
            hourly_missing_values_df = hourly_missing_values_df.append({
                'StudySite': study_site,
                'Hour': hour,
                'TotalValues': total_values,
                **{col: missing_percent[col] for col in group.columns}
            }, ignore_index=True)

        # Calculate missing values per year for each study site
        yearly_groups = df.groupby(df.index.year)
        for year, group in yearly_groups:
            missing_values, missing_percent, total_values = calculate_missing_values(group)
            yearly_missing_values_df = yearly_missing_values_df.append({
                'StudySite': study_site,
                'Year': year,
                'TotalValues': total_values,
                **{col: missing_percent[col] for col in group.columns}
            }, ignore_index=True)

# Save the missing value reports to CSV files
site_missing_values_df.to_csv(os.path.join(output_directory, 'Site_Missing_Values_Report.csv'), index=False)
hourly_missing_values_df.to_csv(os.path.join(output_directory, 'Hourly_Missing_Values_Report.csv'), index=False)
yearly_missing_values_df.to_csv(os.path.join(output_directory, 'Yearly_Missing_Values_Report.csv'), index=False)

print("Missing value reports generated and saved.")


Missing value reports generated and saved.


In [5]:
import os
import pandas as pd

# Define the directory containing the CSV files
directory = '/home/ahmedmas/Projects/Data_imputation/Modified_Data_with_10_percent_missing_PM2.5'
output_directory = '/home/ahmedmas/Projects/Data_imputation/Summery_results_processed3'

# Create the output directory if it does not exist
os.makedirs(output_directory, exist_ok=True)

# Function to extract study site from filename
def extract_study_site(filename):
    parts = filename.split('_')
    study_site = parts[1]  # Assuming the study site is the second word in the filename
    return study_site

# Function to rename columns by extracting the variable name
def rename_columns(df):
    renamed_columns = {}
    study_site = None
    for col in df.columns:
        if col != 'datetime':
            parts = col.split('_')
            if len(parts) > 1:
                var_name = parts[0]
                renamed_columns[col] = var_name
                if not study_site:
                    study_site = parts[1]  # Assuming all columns have the same study site
    df.rename(columns=renamed_columns, inplace=True)
    return study_site

# Function to calculate missing values and percentages
def calculate_missing_values(df):
    missing_values = df.isnull().sum()
    total_values = len(df)
    missing_percent = (missing_values / total_values) * 100
    return missing_values, missing_percent, total_values

# Initialize dataframes to store missing value reports
site_missing_values_df = pd.DataFrame()
hourly_missing_values_df = pd.DataFrame()
yearly_missing_values_df = pd.DataFrame()

# Traverse the directory and process each CSV file
for filename in os.listdir(directory):
    if filename.endswith('.csv'):
        filepath = os.path.join(directory, filename)

        # Load the CSV file into a DataFrame
        df = pd.read_csv(filepath)

        # Convert datetime column to datetime type and sort by datetime
        df['datetime'] = pd.to_datetime(df['datetime'])
        df.sort_values(by='datetime', inplace=True)

        # Extract study site from filename
        study_site = extract_study_site(filename)

        # Rename columns
        rename_columns(df)

        # Set the datetime column as the index
        df.set_index('datetime', inplace=True)

        # Calculate missing values for each study site
        missing_values, missing_percent, total_values = calculate_missing_values(df)
        site_missing_values_df = site_missing_values_df.append({
            'StudySite': study_site,
            'TotalValues': total_values,
            **{col: missing_percent[col] for col in df.columns}
        }, ignore_index=True)

        # Calculate missing values per hour for each study site
        hourly_groups = df.groupby(df.index.hour)
        for hour, group in hourly_groups:
            missing_values, missing_percent, total_values = calculate_missing_values(group)
            hourly_missing_values_df = hourly_missing_values_df.append({
                'StudySite': study_site,
                'Hour': hour,
                'TotalValues': total_values,
                **{col: missing_percent[col] for col in group.columns}
            }, ignore_index=True)

        # Calculate missing values per year for each study site
        yearly_groups = df.groupby(df.index.year)
        for year, group in yearly_groups:
            missing_values, missing_percent, total_values = calculate_missing_values(group)
            yearly_missing_values_df = yearly_missing_values_df.append({
                'StudySite': study_site,
                'Year': year,
                'TotalValues': total_values,
                **{col: missing_percent[col] for col in group.columns}
            }, ignore_index=True)

# Save the missing value reports to CSV files
site_missing_values_df.to_csv(os.path.join(output_directory, 'Site_Missing_Values_Report.csv'), index=False)
hourly_missing_values_df.to_csv(os.path.join(output_directory, 'Hourly_Missing_Values_Report.csv'), index=False)
yearly_missing_values_df.to_csv(os.path.join(output_directory, 'Yearly_Missing_Values_Report.csv'), index=False)

print("Missing value reports generated and saved.")


Missing value reports generated and saved.
